# Prvi model: klasifikacijsko drevo — B: vaja (telekom)
**Akademija UI za poslovne aplikacije — Modul 2, Uvod v strojno učenje**

Predavatelj je v zvezku `A_klasifikacijsko_drevo` pokazal, kako drevo naučimo in preberemo.
Zdaj drevesa spreminjate sami. Na vrhu celice zamenjate vrednost in celico ponovno zaženete (**Shift + Enter**).

### Ponovitev
- **Odločitveno drevo** je zaporedje vprašanj da / ne o značilkah. Na koncu vsake poti je **list** z napovedjo.
- **Učenje:** drevo preizkusi veliko pragov in obdrži tistega, ki najbolje loči stranke, ki ostanejo, od tistih, ki odidejo.
- **Izhodiščni model** vedno napove pogostejši razred. Naš model mora biti boljši od njega.
- **Točnost** (*accuracy*) je delež pravilnih napovedi. Merimo jo na **učni** in na **validacijski** množici.
- **Pretirano prilagajanje:** učna točnost raste, validacijska pa pada. Model si je zapomnil učne podatke.
- **Parametri** so to, kar se model nauči (vprašanja in pragovi). **Hiperparametri** so to, kar nastavimo mi
  (`max_depth`, `min_samples_leaf`).

### Nastavitev okolja
Zaženite spodnji dve celici (**Shift + Enter**). Ničesar ne spreminjajte. Koda prve celice je skrita.
Druga celica pokliče skupno pripravo podatkov iz `utils.modeliranje`. Veljavne pripravljene tabele ponovno uporabi.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


In [ ]:
from utils.modeliranje import poskrbi_za_modeliranje, preberi_shemo, preberi_tabelo
PODATKI = poskrbi_za_modeliranje("telekom")
shema = preberi_shemo(PODATKI / "shema.json")


### Knjižnice in podatki
Funkcije za izris dreves smo napisali v zvezku A; tu jih uvozimo iz `utils/orodja.py`.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from plotnine import *
from sklearn.tree import DecisionTreeClassifier, export_text

from utils.orodja import nastavi_slog, MODRA, ORANZNA, ZELENA, SIVA, SEME
from utils.orodja import narisi_drevo, graf_obmocij, oceni_globine
nastavi_slog()

<details>
<summary><b>Kako so pripravljeni podatki? (klikni)</b></summary>

Podatke je očistila, razdelila in shranila skripta `scripts/pripravi_telekom.py`. Odločitve z razlogi so v zvezkih
`01_priprava_podatkov/A1_Priprava_Telekom` in `02_preverjanje_modela/A1_Delitev_in_manjkajoce_Telekom`. Na kratko:

1. Ena vrstica = ena stranka (100 000 strank). Cilj je `odhod`.
2. Skrite oznake (`U`, `UNKW`) in nemogoče negativne vrednosti so označene kot manjkajoče.
3. Stolpcem iz prvega modula so dodani novi: skupaj 64 značilk (58 številskih, 6 besedilnih).
4. Delitev 60 / 20 / 20: naključna, z enakim deležem odhodov, seme 42. Vsi imamo iste stranke v istih množicah.
5. Manjkajoče številke so zapolnjene z mediano **učne** množice. Pri izbranih stolpcih stolpec s končnico `_manjka` pove, kje je vrednost manjkala.
6. Besedilni stolpci so zapisani tudi kot stolpci 0/1 (na primer `regija_CHICAGO_AREA`).
   Za model je pripravljenih 96 stolpcev: 58 številskih in 38 stolpcev 0/1.
7. Osamelcev ne odstranimo: drevo vpraša samo, ali je vrednost nad ali pod pragom.
8. Modele primerjamo na validacijski množici. Testna ostane zaprta do tekmovanja na koncu klasifikacijskega dela.

</details>

In [ ]:
ucni = preberi_tabelo(PODATKI / "train.csv", shema)
validacijski = preberi_tabelo(PODATKI / "val.csv", shema)
y_ucni, y_val = ucni["odhod"], validacijski["odhod"]

# 16 številskih stolpcev iz prvega modula: z njimi je delal predavatelj v zvezku A
PRVI_MODUL = ["mesecev_narocnine", "povp_mesecni_prihodek", "povp_mesecna_narocnina", "povp_minute",
              "povp_minute_6m", "sprememba_minut", "povp_minute_prekoracitev", "povp_gostovanje",
              "povp_klici_podpora", "povp_prekinjeni_klici", "stevilo_narocnin", "stevilo_telefonov",
              "dni_z_napravo", "cena_naprave", "dohodek_razred", "stevilo_avtomobilov"]
X_ucni, X_val = ucni[PRVI_MODUL], validacijski[PRVI_MODUL]

izhodisce = (y_val == y_ucni.mode()[0]).mean()
print(f"Učnih strank: {len(X_ucni)}, validacijskih: {len(X_val)}, točnost izhodišča: {izhodisce:.3f}")

---
## B1. Spremenite globino
Poskusite `GLOBINA = 1, 2, 3, 4, 5`.

**Vprašanja:**
1. Katero vprašanje postavi drevo najprej? Se to spremeni, ko povečate globino?
2. Pri globini 3: v katerem listu je največji delež strank, ki odidejo? Koliko strank je v njem?
3. Za koliko se validacijska točnost izboljša od globine 1 do globine 5?

In [ ]:
GLOBINA = 2          # ← zamenjajte

drevo = DecisionTreeClassifier(max_depth=GLOBINA, random_state=SEME).fit(X_ucni, y_ucni)

narisi_drevo(drevo, PRVI_MODUL)
plt.show()
print(f"Listov: {drevo.get_n_leaves()}   učna točnost: {drevo.score(X_ucni, y_ucni):.3f}   "
      f"validacijska: {drevo.score(X_val, y_val):.3f}   (izhodišče {izhodisce:.3f})")

<details>
<summary><b>Rešitev (klikni)</b></summary>

1. Pri preizkušenih globinah 1–5 je prvo vprašanje vedno `dni_z_napravo <= 304,5`. Drevo pri večji globini doda vprašanja nižje; pri drugih nastavitvah bi se lahko spremenilo tudi prvo vprašanje.
2. List desno: `dni_z_napravo > 304,5` in `povp_minute <= 0,4` in `povp_minute_6m > 4,5`.
   V njem je 391 strank, odide jih 88 %. To so stranke s starim telefonom, ki so prej klicale, zdaj pa skoraj ne kličejo več.
3. Validacijska točnost se med preizkušenima globinama poveča z 0,581 na 0,596. Dodatna vprašanja pomagajo, vendar tu manj kot prvo.

</details>

Preverimo: točnost za globine 1–5.

In [ ]:
oceni_globine(X_ucni, y_ucni, X_val, y_val, globine=range(1, 6)).round(3)

## Katere stolpce lahko izberete?
Do zdaj je drevo videlo 16 stolpcev iz prvega modula (`PRVI_MODUL`). V tabeli je za model pripravljenih **96 stolpcev**.
Razdeljeni so v skupine. Besedilni stolpci so zapisani kot stolpci 0/1: `regija_CHICAGO_AREA` je 1, če je stranka iz te regije.

Seznam stolpcev lahko sestavite na tri načine:
- cela skupina ali več skupin: `SKUPINE["naprava"] + SKUPINE["prihodki"]`
- vsi stolpci: `VSE_ZNACILKE`
- posamezni stolpci: `["dni_z_napravo", "mesecev_narocnine", "cena_naprave"]`

In [ ]:
shema = preberi_shemo(PODATKI / "shema.json")     # opis stolpcev, ki ga je shranila skripta
SKUPINE = shema["skupine_za_model"]              # skupina -> seznam stolpcev
VSE_ZNACILKE = shema["znacilke_za_model"]        # vseh 96 stolpcev, pripravljenih za model

pd.DataFrame([{"skupina": ime, "stolpcev": len(stolpci), "prvi trije stolpci": ", ".join(stolpci[:3])}
              for ime, stolpci in SKUPINE.items()])

Vse stolpce ene skupine izpišete tako (zamenjajte ime skupine):

In [ ]:
SKUPINE["naprava"]          # ← zamenjajte ime skupine

## B2. Naloga v paru: previdno in drzno drevo
Pare smo določili na začetku vaje: v vsakem paru je eden **levi** in eden **desni**. Tako ostane ves dan.
Vsak nauči svoje drevo. Razlikujeta se v treh nastavitvah:

| | `ZNACILKE` (stolpci) | `GLOBINA` (največ vprašanj zapored) | `NAJMANJ_V_LISTU` (najmanj strank v listu) |
|---|---|---|---|
| **levi v paru: previdno drevo** | `PRVI_MODUL` (16 stolpcev) | 3 | 500 |
| **desni v paru: drzno drevo** | `VSE_ZNACILKE` (96 stolpcev) | 20 | 1 |

**1. korak.** Zaženita celico in primerjajta rezultate. Zemljevid prikazuje **ločeno drevo z istima nastavitvama globine in lista**,
naučeno samo na dveh značilkah.

**2. korak.** Kaj je naredilo razliko: stolpci ali globina? Oba zamenjajta **samo seznam stolpcev**
(levi vzame `VSE_ZNACILKE`, desni `PRVI_MODUL`) in celico zaženita še enkrat.

| | stolpcev | listov | učna točnost | validacijska točnost |
|---|---|---|---|---|
| previdno (3 in 500), `PRVI_MODUL` | 16 | | | |
| drzno (20 in 1), `VSE_ZNACILKE` | 96 | | | |
| previdno (3 in 500), `VSE_ZNACILKE` | 96 | | | |
| drzno (20 in 1), `PRVI_MODUL` | 16 | | | |

**Vprašanja:** Kdo zmaga na učnih podatkih? Kdo na validacijskih? Ali več stolpcev pomeni boljše drevo?
Kateremu drevesu bi zaupali pri novih strankah?

In [ ]:
ZNACILKE = PRVI_MODUL      # ← stolpci: PRVI_MODUL ali VSE_ZNACILKE
GLOBINA = 3                # ← največ vprašanj zapored (max_depth): 3 ali 20
NAJMANJ_V_LISTU = 500      # ← najmanj strank v listu (min_samples_leaf): 500 ali 1

drevo = DecisionTreeClassifier(max_depth=GLOBINA, min_samples_leaf=NAJMANJ_V_LISTU, random_state=SEME).fit(ucni[ZNACILKE], y_ucni)
print(f"Stolpcev: {len(ZNACILKE)}   listov: {drevo.get_n_leaves()}   učna točnost: {drevo.score(ucni[ZNACILKE], y_ucni):.3f}   "
      f"validacijska: {drevo.score(validacijski[ZNACILKE], y_val):.3f}")

DVE = ["dni_z_napravo", "mesecev_narocnine"]
drevo_dve = DecisionTreeClassifier(max_depth=GLOBINA, min_samples_leaf=NAJMANJ_V_LISTU, random_state=SEME).fit(X_ucni[DVE], y_ucni)
graf_obmocij(drevo_dve, X_ucni, *DVE) + labs(title=f"Zemljevid: globina {GLOBINA}, najmanj {NAJMANJ_V_LISTU} strank v listu")

<details>
<summary><b>Rešitev (klikni)</b></summary>

| | stolpcev | listov | učna točnost | validacijska točnost |
|---|---|---|---|---|
| previdno (3 in 500), `PRVI_MODUL` | 16 | 7 | 0,589 | 0,589 |
| drzno (20 in 1), `VSE_ZNACILKE` | 96 | 6 896 | 0,890 | 0,562 |
| previdno (3 in 500), `VSE_ZNACILKE` | 96 | 8 | 0,589 | 0,589 |
| drzno (20 in 1), `PRVI_MODUL` | 16 | 7 854 | 0,864 | 0,553 |

- Na učnih podatkih zmaga drzno drevo, na validacijskih previdno.
- Pri previdnem drevesu sta zaokroženi učna in validacijska točnost podobni (0,589). To je dober znak na tej delitvi, ne dokaz, da bo enako pri vseh prihodnjih strankah.
- Pri drznem je razlika velika: na učnih podatkih zadene veliko več kot na validacijskih. Majhni listi ujamejo podrobnosti učnih strank, ki se na novih ne ponovijo.
- Pri teh dveh previdnih drevesih dodanih 80 stolpcev ni izboljšalo validacijske točnosti. Iz tabele ne moremo sklepati, da dodatni stolpci nikoli ne pomagajo.
- Med temi kandidati izberemo previdno drevo po **validacijski** točnosti; testna množica ostane za končno oceno.

</details>

Preverimo vse štiri kombinacije naenkrat:

In [ ]:
vrstice = []
for ime, globina, najmanj in [("previdno (3 in 500)", 3, 500), ("drzno (20 in 1)", 20, 1)]:
    for ime_stolpcev, stolpci in [("PRVI_MODUL", PRVI_MODUL), ("VSE_ZNACILKE", VSE_ZNACILKE)]:
        d = DecisionTreeClassifier(max_depth=globina, min_samples_leaf=najmanj, random_state=SEME).fit(ucni[stolpci], y_ucni)
        vrstice.append({"drevo": ime, "stolpci": ime_stolpcev, "stolpcev": len(stolpci), "listov": d.get_n_leaves(),
                        "učna": d.score(ucni[stolpci], y_ucni), "validacijska": d.score(validacijski[stolpci], y_val)})
pd.DataFrame(vrstice).round(3)

## B3. Poiščite najboljšo globino
Celica nauči 20 dreves (globine 1–20) na 16 stolpcih iz prvega modula in nariše obe krivulji. Traja nekaj sekund.

**Vprašanje:** Pri kateri globini je validacijska točnost največja? Kaj se dogaja desno od vrha?

In [ ]:
rezultati = oceni_globine(X_ucni, y_ucni, X_val, y_val, globine=range(1, 21))
dolga = rezultati.melt(id_vars="globina", value_vars=["učna", "validacijska"], var_name="množica", value_name="točnost")

(ggplot(dolga, aes(x="globina", y="točnost", color="množica"))
 + geom_line(size=1.2)
 + geom_point(size=2)
 + scale_color_manual(values={"učna": MODRA, "validacijska": ORANZNA})
 + labs(title="Točnost glede na globino drevesa", x="globina (max_depth)", y="točnost", color=""))

<details>
<summary><b>Rešitev (klikni)</b></summary>

Na teh 20 000 validacijskih strankah je najvišja točnost pri globini 8 (0,606). Pri večjih globinah učna točnost
še raste, validacijska pa pada: drevo vse bolje opisuje učne stranke, novih pa ne napoveduje bolje.
To je znak pretiranega prilagajanja. Globino izberemo na validaciji; testna množica ostane za končno oceno.

</details>

Preverimo: tri globine z najvišjo validacijsko točnostjo.

In [ ]:
rezultati.sort_values("validacijska", ascending=False).head(3).round(3)

## B4. Odstranite najljubšo značilko
Drevo vedno najprej vpraša po `dni_z_napravo`. Kaj vpraša, če mu te značilke ne damo? Izhajamo iz 16 stolpcev `PRVI_MODUL`.
Nato v seznam `BREZ` dodajte še `"mesecev_narocnine"`.

In [ ]:
BREZ = ["dni_z_napravo"]     # ← dodajte še "mesecev_narocnine"

izbrane = [z for z in PRVI_MODUL if z not in BREZ]
drevo = DecisionTreeClassifier(max_depth=2, random_state=SEME).fit(X_ucni[izbrane], y_ucni)

narisi_drevo(drevo, izbrane, velikost=(10, 4))
plt.show()
print(f"Validacijska točnost: {drevo.score(X_val[izbrane], y_val):.3f}   (z vsemi 16 značilkami, globina 2: 0,581)")

<details>
<summary><b>Rešitev (klikni)</b></summary>

- Brez `dni_z_napravo` drevo najprej vpraša `mesecev_narocnine <= 10,5`. Validacijska točnost pade z 0,581 na 0,558.
- Brez obeh najprej vpraša `cena_naprave <= 70`. Validacijska točnost je 0,554.

Na tej validacijski množici je model še vedno boljši od izhodišča (0,504). Del informacije je tudi v drugih značilkah:
ko `dni_z_napravo` odstranimo, jih drevo uporabi namesto nje, čeprav je točnost nižja. Cenejše naprave in več odhodov smo videli že v 1. modulu.

</details>

Preverimo: prvo vprašanje in validacijska točnost za oba primera.

In [ ]:
for brez in [["dni_z_napravo"], ["dni_z_napravo", "mesecev_narocnine"]]:
    izbrane = [z for z in PRVI_MODUL if z not in brez]
    d = DecisionTreeClassifier(max_depth=2, random_state=SEME).fit(X_ucni[izbrane], y_ucni)
    prvo = f"{izbrane[d.tree_.feature[0]]} <= {d.tree_.threshold[0]:.1f}"
    print(f"brez {brez}: prvo vprašanje {prvo}, validacijska točnost {d.score(X_val[izbrane], y_val):.3f}")

## B5. Moj kandidat za tekmovanje
Sestavite drevo, ki ga boste prijavili na tekmovanje. Izberite stolpce in nastavitve, celico zaženite večkrat
in obdržite različico z najvišjo **validacijsko** točnostjo. Zadnji izpis si shranite: z njim boste na tekmovanju ponovili svoj model.

Testno množico odpremo samo enkrat: v zvezku za tekmovanje, na koncu klasifikacijskega dela.

In [ ]:
ZNACILKE = PRVI_MODUL          # ← stolpci: PRVI_MODUL, VSE_ZNACILKE, SKUPINE["naprava"] + SKUPINE["prihodki"] ali svoj seznam
GLOBINA = 5                    # ← največ vprašanj zapored (max_depth)
NAJMANJ_V_LISTU = 200          # ← najmanj strank v listu (min_samples_leaf)
SEME_DREVESA = 42              # ← seme naključja (random_state)

drevo = DecisionTreeClassifier(max_depth=GLOBINA, min_samples_leaf=NAJMANJ_V_LISTU, random_state=SEME_DREVESA).fit(ucni[ZNACILKE], y_ucni)
tocnost = drevo.score(validacijski[ZNACILKE], y_val)
print(f"Stolpcev: {len(ZNACILKE)}   listov: {drevo.get_n_leaves()}   učna točnost: {drevo.score(ucni[ZNACILKE], y_ucni):.3f}   "
      f"validacijska: {tocnost:.3f}   (izhodišče {izhodisce:.3f})")

kandidat = {"model": "drevo", "znacilke": list(ZNACILKE),
            "nastavitve": {"max_depth": GLOBINA, "min_samples_leaf": NAJMANJ_V_LISTU},
            "seme": SEME_DREVESA, "tocnost_validacija": round(tocnost, 4)}
print(kandidat)

---
### Za hitre: vaša stranka
Spremenite vrednosti in poglejte, kaj napove drevo globine 3.

In [ ]:
drevo_3 = DecisionTreeClassifier(max_depth=3, random_state=SEME).fit(X_ucni, y_ucni)

moja_stranka = X_val.iloc[[0]].copy()               # začnemo z resnično stranko
moja_stranka["dni_z_napravo"] = 100                 # ← spremenite
moja_stranka["mesecev_narocnine"] = 12              # ← spremenite
moja_stranka["povp_minute"] = 300                   # ← spremenite

print("Verjetnost odhoda:", drevo_3.predict_proba(moja_stranka)[0, 1].round(2))